# DS3/GSE182109 — dataset-level quality control

This notebook reproduces the DS3 filtering rules used in the study.

## Cell filters

Primary cell retention requires:

- detected genes > 500;
- detected genes < 8,000;
- total counts > 1,000;
- total counts < 60,000;
- mitochondrial fraction < 12%.

A final total-count ceiling of 45,000 is then applied, followed by
retention of genes detected in at least 10 final cells.

## Expected dimensions

- input: 264,951 cells × 38,224 genes
- after primary cell filtering: 219,418 cells × 38,224 genes
- final post-QC: 218,735 cells × 30,284 genes

Raw counts remain in `X` and are duplicated into `layers["counts"]` in
the final file. No normalization or log transformation is performed.

## Memory-safe implementation

The input is opened in backed mode and processed in bounded row chunks.
The final object is assembled on disk from filtered chunks, so the full
input matrix and a second multi-gigabyte copy are never held in RAM at
the same time. The `counts` layer is copied directly inside the HDF5
file after on-disk concatenation.

Input:

`data/interim/DS3_GSE182109/GSE182109_merged_rawcounts.h5ad`

Output:

`data/processed/DS3_GSE182109/GSE182109_postQC.h5ad`


In [1]:
import gc
import json
import os
import shutil
import sys
from copy import deepcopy
from datetime import datetime
from pathlib import Path

import anndata as ad
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

try:
    from anndata.experimental import concat_on_disk
except ImportError as exc:
    raise ImportError(
        "This notebook requires anndata.experimental.concat_on_disk. "
        "Install a current anndata release in the glioma-cnv-atlas environment."
    ) from exc

print("Python:", sys.version)
print("anndata:", ad.__version__)
print("scanpy:", sc.__version__)
print("pandas:", pd.__version__)


Python: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]
anndata: 0.12.7
scanpy: 1.11.5
pandas: 2.3.3


C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_28896\3482966472.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)
C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_28896\3482966472.py:28: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)


In [2]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

INPUT_H5AD = Path(
    os.environ.get(
        "GSE182109_MERGED_H5AD",
        PROJECT_ROOT
        / "data"
        / "interim"
        / "DS3_GSE182109"
        / "GSE182109_merged_rawcounts.h5ad",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE182109_POSTQC_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS3_GSE182109"
        / "GSE182109_postQC.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = PROJECT_ROOT / "results" / "qc" / "DS3_GSE182109" / "00b_qc"
FIGURE_DIR = AUDIT_DIR / "figures"

EXPECTED_INPUT_SHAPE = (264_951, 38_224)
EXPECTED_AFTER_PRIMARY_FILTER = 219_418
EXPECTED_OUTPUT_SHAPE = (218_735, 30_284)

MIN_GENES_EXCLUSIVE = 500
MAX_GENES_EXCLUSIVE = 8_000
MIN_COUNTS_EXCLUSIVE = 1_000
MAX_COUNTS_PRIMARY_EXCLUSIVE = 60_000
MAX_MT_PERCENT_EXCLUSIVE = 12.0
FINAL_MAX_COUNTS_INCLUSIVE = 45_000
FINAL_MIN_CELLS_PER_GENE = 10

# Conservative values for a 16-GB Windows workstation.
QC_ROW_CHUNK_SIZE = 2_048
OUTPUT_SOURCE_ROW_CHUNK_SIZE = 4_096
CONCAT_MAX_LOADED_ELEMS = 20_000_000
PLOT_MAX_CELLS = 50_000
RANDOM_SEED = 0

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", INPUT_H5AD)
print("Output:", OUTPUT_H5AD)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Merged input not found: {INPUT_H5AD}\n"
        "Run 00a_ds3_build_raw_anndata.ipynb first or set "
        "GSE182109_MERGED_H5AD."
    )


Input: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\interim\DS3_GSE182109\GSE182109_merged_rawcounts.h5ad
Output: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\processed\DS3_GSE182109\GSE182109_postQC.h5ad


In [3]:
def matrix_values(matrix):
    return matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()


def assert_raw_counts(matrix, label: str):
    values = matrix_values(matrix)
    if values.size == 0:
        raise ValueError(f"{label} contains no nonzero values.")
    if not np.isfinite(values).all():
        raise ValueError(f"{label} contains non-finite values.")
    if np.any(values < 0):
        raise ValueError(f"{label} contains negative values.")
    if not np.equal(values, np.floor(values)).all():
        raise ValueError(f"{label} is not an integer raw-count matrix.")


def save_current_figure(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close("all")


def read_csr_rows(backed_adata, row_start: int, row_end: int) -> sp.csr_matrix:
    """Materialize one contiguous row block from a backed sparse matrix."""
    block = backed_adata.X[row_start:row_end, :]
    if not sp.issparse(block):
        block = sp.csr_matrix(block)
    elif not sp.isspmatrix_csr(block):
        block = block.tocsr()
    else:
        block = block.copy()

    block.sum_duplicates()
    block.eliminate_zeros()
    return block


def percentage(numerator: np.ndarray, denominator: np.ndarray) -> np.ndarray:
    result = np.zeros(denominator.shape, dtype=np.float64)
    np.divide(
        numerator * 100.0,
        denominator,
        out=result,
        where=denominator > 0,
    )
    return result


In [4]:
# Open the large matrix in backed mode. obs, var, and uns remain small enough
# to keep in memory; X stays on disk and is accessed by row chunks.
adata = sc.read_h5ad(INPUT_H5AD, backed="r")

if tuple(adata.shape) != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        f"Expected input shape {EXPECTED_INPUT_SHAPE}, observed {adata.shape}"
    )

required_obs = {"gsm", "sample", "group", "patient", "region", "sample_key"}
missing_obs = sorted(required_obs - set(adata.obs.columns))
if missing_obs:
    raise KeyError(f"Required metadata columns are missing: {missing_obs}")

if not adata.obs_names.is_unique:
    raise AssertionError("Cell identifiers are not unique.")
if not adata.var_names.is_unique:
    raise AssertionError("Gene identifiers are not unique.")

input_obs = adata.obs.copy()
input_var = adata.var.copy()
input_uns = deepcopy(dict(adata.uns))
input_obs_names = adata.obs_names.to_numpy(dtype=str)
input_var_names = adata.var_names.to_numpy(dtype=str)

print("Backed input:", adata)
print("Backed matrix type:", type(adata.X).__name__)


Backed input: AnnData object with n_obs × n_vars = 264951 × 38224 backed at 'C:\\Users\\AmirSilco\\Documents\\GitHub\\glioma-cnv-single-cell-atlas\\data\\interim\\DS3_GSE182109\\GSE182109_merged_rawcounts.h5ad'
    obs: 'cell_barcode', 'gsm', 'sample', 'sample_id', 'group', 'patient', 'region', 'sample_key', 'geo_id', 'core_dataset'
    var: 'gene_ids', 'feature_types', 'feature_name_original'
    uns: 'build_timestamp', 'dataset_id', 'feature_join', 'feature_metadata_conflict_count', 'geo_accession', 'source_sample_count'
Backed matrix type: _CSRDataset


In [5]:
print("Running DS3 chunked QC metrics")

# ------------------------------------------------------------------
# Feature-class masks
# ------------------------------------------------------------------
feature_column_candidates = [
    "feature_name_original",
    "gene_symbols",
    "gene_symbol",
]
feature_column = next(
    (column for column in feature_column_candidates if column in input_var.columns),
    None,
)

if feature_column is None:
    feature_names = pd.Series(input_var_names, index=input_var.index, dtype="string")
    print("Feature names taken from var_names")
else:
    feature_names = input_var[feature_column].astype("string")
    print(f"Feature names taken from var[{feature_column!r}]")

feature_upper = feature_names.fillna("").str.strip().str.upper()
mt_mask = feature_upper.str.startswith("MT-", na=False).to_numpy(dtype=bool)
ribo_mask = feature_upper.str.startswith(("RPS", "RPL"), na=False).to_numpy(dtype=bool)
hb_mask = feature_upper.str.match(r"^HB[ABDEGQMZ]", na=False).to_numpy(dtype=bool)

input_var["mt"] = mt_mask
input_var["ribo"] = ribo_mask
input_var["hb"] = hb_mask

# ------------------------------------------------------------------
# Cell-level and gene-level QC metrics in bounded row chunks
# ------------------------------------------------------------------
n_obs, n_vars = EXPECTED_INPUT_SHAPE
n_genes_by_counts = np.zeros(n_obs, dtype=np.int32)
total_counts = np.zeros(n_obs, dtype=np.float64)
mt_counts = np.zeros(n_obs, dtype=np.float64)
ribo_counts = np.zeros(n_obs, dtype=np.float64)
hb_counts = np.zeros(n_obs, dtype=np.float64)

n_cells_by_counts = np.zeros(n_vars, dtype=np.int64)
total_counts_by_gene = np.zeros(n_vars, dtype=np.float64)

for row_start in range(0, n_obs, QC_ROW_CHUNK_SIZE):
    row_end = min(row_start + QC_ROW_CHUNK_SIZE, n_obs)
    block = read_csr_rows(adata, row_start, row_end)
    assert_raw_counts(block, f"adata.X[{row_start}:{row_end}, :]")

    n_genes_by_counts[row_start:row_end] = np.diff(block.indptr).astype(
        np.int32, copy=False
    )
    block_totals = np.asarray(block.sum(axis=1)).ravel().astype(np.float64)
    total_counts[row_start:row_end] = block_totals

    if mt_mask.any():
        mt_counts[row_start:row_end] = np.asarray(
            block[:, mt_mask].sum(axis=1)
        ).ravel()
    if ribo_mask.any():
        ribo_counts[row_start:row_end] = np.asarray(
            block[:, ribo_mask].sum(axis=1)
        ).ravel()
    if hb_mask.any():
        hb_counts[row_start:row_end] = np.asarray(
            block[:, hb_mask].sum(axis=1)
        ).ravel()

    # Bounded bincounts avoid Scanpy/SciPy's multi-gigabyte conversion of
    # every sparse column index in the complete matrix at once.
    n_cells_by_counts += np.bincount(block.indices, minlength=n_vars)
    total_counts_by_gene += np.bincount(
        block.indices,
        weights=block.data,
        minlength=n_vars,
    )

    if row_end == n_obs or row_end % 20_000 < QC_ROW_CHUNK_SIZE:
        print(f"QC rows processed: {row_end:,}/{n_obs:,}")

    del block, block_totals
    gc.collect()

pct_counts_mt = percentage(mt_counts, total_counts)
pct_counts_ribo = percentage(ribo_counts, total_counts)
pct_counts_hb = percentage(hb_counts, total_counts)

input_obs["n_genes_by_counts"] = n_genes_by_counts
input_obs["log1p_n_genes_by_counts"] = np.log1p(n_genes_by_counts)
input_obs["total_counts"] = total_counts
input_obs["log1p_total_counts"] = np.log1p(total_counts)
input_obs["total_counts_mt"] = mt_counts
input_obs["log1p_total_counts_mt"] = np.log1p(mt_counts)
input_obs["pct_counts_mt"] = pct_counts_mt
input_obs["total_counts_ribo"] = ribo_counts
input_obs["log1p_total_counts_ribo"] = np.log1p(ribo_counts)
input_obs["pct_counts_ribo"] = pct_counts_ribo
input_obs["total_counts_hb"] = hb_counts
input_obs["log1p_total_counts_hb"] = np.log1p(hb_counts)
input_obs["pct_counts_hb"] = pct_counts_hb

mean_counts = total_counts_by_gene / float(n_obs)
input_var["n_cells_by_counts"] = n_cells_by_counts
input_var["mean_counts"] = mean_counts
input_var["log1p_mean_counts"] = np.log1p(mean_counts)
input_var["pct_dropout_by_counts"] = (
    1.0 - n_cells_by_counts.astype(np.float64) / float(n_obs)
) * 100.0
input_var["total_counts"] = total_counts_by_gene
input_var["log1p_total_counts"] = np.log1p(total_counts_by_gene)

required_qc_columns = [
    "n_genes_by_counts",
    "total_counts",
    "pct_counts_mt",
    "pct_counts_ribo",
    "pct_counts_hb",
]
for column in required_qc_columns:
    values = pd.to_numeric(input_obs[column], errors="raise").to_numpy()
    if not np.isfinite(values).all():
        raise ValueError(f"obs[{column!r}] contains non-finite values.")

print("Chunked QC metrics completed.")
print("Mitochondrial features:", int(mt_mask.sum()))
print("Ribosomal features:", int(ribo_mask.sum()))
print("Hemoglobin features:", int(hb_mask.sum()))
input_obs[required_qc_columns].head()


Running DS3 chunked QC metrics
Feature names taken from var['feature_name_original']
QC rows processed: 20,480/264,951
QC rows processed: 40,960/264,951
QC rows processed: 61,440/264,951
QC rows processed: 81,920/264,951
QC rows processed: 100,352/264,951
QC rows processed: 120,832/264,951
QC rows processed: 141,312/264,951
QC rows processed: 161,792/264,951
QC rows processed: 180,224/264,951
QC rows processed: 200,704/264,951
QC rows processed: 221,184/264,951
QC rows processed: 241,664/264,951
QC rows processed: 260,096/264,951
QC rows processed: 264,951/264,951
Chunked QC metrics completed.
Mitochondrial features: 13
Ribosomal features: 105
Hemoglobin features: 11


,n_genes_by_counts,total_counts,pct_counts_mt,pct_counts_ribo,pct_counts_hb
cell_id,,,,,
AAACCTGAGAAGCCCA-1-GSM5518596_rGBM-01-A,5326,17319.0,4.186154,7.985450,0.011548
AAACCTGCACTGTCGG-1-GSM5518596_rGBM-01-A,2551,7808.0,0.409836,21.157787,0.012807
AAACCTGCAGCGTCCA-1-GSM5518596_rGBM-01-A,2179,6125.0,0.489796,21.975510,0.000000
AAACCTGGTAAATGAC-1-GSM5518596_rGBM-01-A,3781,17177.0,2.363626,18.064854,0.000000
AAACCTGGTCACTTCC-1-GSM5518596_rGBM-01-A,3156,14139.0,0.169743,22.745597,0.000000


In [6]:
print("Applying DS3 QC rules and calculating the final gene mask")

primary_masks = pd.DataFrame(index=input_obs.index)
primary_masks["pass_min_genes"] = (
    input_obs["n_genes_by_counts"].to_numpy() > MIN_GENES_EXCLUSIVE
)
primary_masks["pass_max_genes"] = (
    input_obs["n_genes_by_counts"].to_numpy() < MAX_GENES_EXCLUSIVE
)
primary_masks["pass_min_counts"] = (
    input_obs["total_counts"].to_numpy() > MIN_COUNTS_EXCLUSIVE
)
primary_masks["pass_max_counts_primary"] = (
    input_obs["total_counts"].to_numpy() < MAX_COUNTS_PRIMARY_EXCLUSIVE
)
primary_masks["pass_max_mt"] = (
    input_obs["pct_counts_mt"].to_numpy() < MAX_MT_PERCENT_EXCLUSIVE
)
primary_masks["pass_primary_QC"] = primary_masks.all(axis=1)

primary_cell_mask = primary_masks["pass_primary_QC"].to_numpy(dtype=bool)
observed_primary_cells = int(primary_cell_mask.sum())
if observed_primary_cells != EXPECTED_AFTER_PRIMARY_FILTER:
    raise AssertionError(
        "Primary QC did not reproduce the reported cell count. "
        f"Expected {EXPECTED_AFTER_PRIMARY_FILTER}, observed "
        f"{observed_primary_cells}."
    )

final_cell_mask = primary_cell_mask & (
    input_obs["total_counts"].to_numpy(dtype=np.float64)
    <= FINAL_MAX_COUNTS_INCLUSIVE
)
observed_final_cells = int(final_cell_mask.sum())
if observed_final_cells != EXPECTED_OUTPUT_SHAPE[0]:
    raise AssertionError(
        "Final cell ceiling did not reproduce the reported cell count. "
        f"Expected {EXPECTED_OUTPUT_SHAPE[0]}, observed {observed_final_cells}."
    )

rule_summary = pd.DataFrame(
    {
        "rule": primary_masks.columns,
        "cells_passing": [
            int(primary_masks[column].sum()) for column in primary_masks.columns
        ],
        "cells_failing": [
            int((~primary_masks[column]).sum()) for column in primary_masks.columns
        ],
    }
)
rule_summary.to_csv(
    AUDIT_DIR / "GSE182109_primary_QC_rule_summary.tsv",
    sep="\t",
    index=False,
)

# Count final-cell detection per gene without materializing the filtered object.
final_n_cells_by_gene = np.zeros(EXPECTED_INPUT_SHAPE[1], dtype=np.int64)
for row_start in range(0, EXPECTED_INPUT_SHAPE[0], QC_ROW_CHUNK_SIZE):
    row_end = min(row_start + QC_ROW_CHUNK_SIZE, EXPECTED_INPUT_SHAPE[0])
    local_keep = final_cell_mask[row_start:row_end]
    if not local_keep.any():
        continue

    block = read_csr_rows(adata, row_start, row_end)[local_keep, :].tocsr()
    block.sum_duplicates()
    block.eliminate_zeros()
    final_n_cells_by_gene += np.bincount(
        block.indices,
        minlength=EXPECTED_INPUT_SHAPE[1],
    )
    del block
    gc.collect()

gene_keep_mask = final_n_cells_by_gene >= FINAL_MIN_CELLS_PER_GENE
observed_final_genes = int(gene_keep_mask.sum())
if observed_final_genes != EXPECTED_OUTPUT_SHAPE[1]:
    raise AssertionError(
        "Final gene filter did not reproduce the reported gene count. "
        f"Expected {EXPECTED_OUTPUT_SHAPE[1]}, observed {observed_final_genes}."
    )

final_obs = input_obs.iloc[np.flatnonzero(final_cell_mask)].copy()
final_obs["n_counts"] = final_obs["total_counts"].astype(np.float32)
final_var = input_var.iloc[np.flatnonzero(gene_keep_mask)].copy()
final_var["n_cells_after_QC"] = final_n_cells_by_gene[gene_keep_mask]

before_sample_counts = (
    input_obs.groupby("sample", observed=True).size().rename("cells_before_QC")
)
after_sample_counts = (
    final_obs.groupby("sample", observed=True).size().rename("cells_after_QC")
)
sample_retention = pd.concat(
    [before_sample_counts, after_sample_counts], axis=1
).fillna(0)
sample_retention["cells_removed"] = (
    sample_retention["cells_before_QC"] - sample_retention["cells_after_QC"]
)
sample_retention["retention_fraction"] = (
    sample_retention["cells_after_QC"] / sample_retention["cells_before_QC"]
)
sample_retention.to_csv(
    AUDIT_DIR / "GSE182109_QC_retention_by_sample.tsv",
    sep="\t",
)

qc_summary = {
    "dataset_id": "DS3_GSE182109",
    "geo_accession": "GSE182109",
    "input_cells": int(EXPECTED_INPUT_SHAPE[0]),
    "input_genes": int(EXPECTED_INPUT_SHAPE[1]),
    "post_primary_filter_cells": observed_primary_cells,
    "post_QC_cells": observed_final_cells,
    "post_QC_genes": observed_final_genes,
    "retention_fraction": float(observed_final_cells / EXPECTED_INPUT_SHAPE[0]),
    "min_genes_exclusive": MIN_GENES_EXCLUSIVE,
    "max_genes_exclusive": MAX_GENES_EXCLUSIVE,
    "min_counts_exclusive": MIN_COUNTS_EXCLUSIVE,
    "primary_max_counts_exclusive": MAX_COUNTS_PRIMARY_EXCLUSIVE,
    "max_mt_percent_exclusive": MAX_MT_PERCENT_EXCLUSIVE,
    "final_max_counts_inclusive": FINAL_MAX_COUNTS_INCLUSIVE,
    "final_min_cells_per_gene": FINAL_MIN_CELLS_PER_GENE,
    "raw_counts_in_X": True,
    "raw_counts_layer": "counts",
    "normalization_applied": False,
    "memory_safe_chunked_processing": True,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "output_h5ad": str(OUTPUT_H5AD),
}
output_uns = input_uns
output_uns["dataset_level_QC"] = qc_summary

print("Primary-filter cells:", observed_primary_cells)
print("Final cells:", observed_final_cells)
print("Final genes:", observed_final_genes)
sample_retention


Applying DS3 QC rules and calculating the final gene mask
Primary-filter cells: 219418
Final cells: 218735
Final genes: 30284


,cells_before_QC,cells_after_QC,cells_removed,retention_fraction
sample,,,,
GSM5518596_rGBM-01-A,4594,4324,270,0.941228
GSM5518597_rGBM-01-B,11716,10960,756,0.935473
GSM5518598_rGBM-01-C,5965,5653,312,0.947695
GSM5518599_rGBM-01-D,6434,5992,442,0.931302
GSM5518600_ndGBM-01-A,5018,4631,387,0.922878
GSM5518601_ndGBM-01-C,10342,9470,872,0.915684
GSM5518602_ndGBM-01-D,9338,8770,568,0.939173
GSM5518603_ndGBM-01-F,5773,4962,811,0.859518
GSM5518604_ndGBM-11-A,805,374,431,0.464596


In [8]:
print("Assembling the final DS3 object on disk")

staging_dir = OUTPUT_H5AD.parent / f".{OUTPUT_H5AD.stem}_chunk_staging"
temporary_output = OUTPUT_H5AD.with_name(
    f"{OUTPUT_H5AD.stem}.temporary{OUTPUT_H5AD.suffix}"
)

if staging_dir.exists():
    shutil.rmtree(staging_dir)
staging_dir.mkdir(parents=True, exist_ok=False)

if temporary_output.exists():
    temporary_output.unlink()

free_bytes = shutil.disk_usage(OUTPUT_H5AD.parent).free
minimum_free_bytes = max(
    20 * 1024**3,
    int(INPUT_H5AD.stat().st_size * 3.0),
)
print("Free disk space:", f"{free_bytes / 1024**3:.2f} GiB")
print("Required free-space guard:", f"{minimum_free_bytes / 1024**3:.2f} GiB")
if free_bytes < minimum_free_bytes:
    raise OSError(
        "Insufficient free disk space for chunk staging, on-disk "
        "concatenation, and the counts-layer copy."
    )

chunk_paths = []
written_cells = 0
n_obs = EXPECTED_INPUT_SHAPE[0]

try:
    for row_start in range(0, n_obs, OUTPUT_SOURCE_ROW_CHUNK_SIZE):
        row_end = min(row_start + OUTPUT_SOURCE_ROW_CHUNK_SIZE, n_obs)
        local_keep = final_cell_mask[row_start:row_end]
        if not local_keep.any():
            continue

        block = read_csr_rows(adata, row_start, row_end)
        filtered_block = block[local_keep, :][:, gene_keep_mask].tocsr()
        filtered_block.sum_duplicates()
        filtered_block.eliminate_zeros()
        assert_raw_counts(
            filtered_block,
            f"final chunk from input rows {row_start}:{row_end}",
        )

        global_positions = row_start + np.flatnonzero(local_keep)
        obs_chunk = input_obs.iloc[global_positions].copy()
        obs_chunk["n_counts"] = obs_chunk["total_counts"].astype(np.float32)

        chunk_adata = ad.AnnData(
            X=filtered_block,
            obs=obs_chunk,
            var=final_var.copy(),
            uns=deepcopy(output_uns),
        )

        chunk_path = staging_dir / f"chunk_{len(chunk_paths):04d}.h5ad"
        chunk_adata.write_h5ad(chunk_path, compression="lzf")
        chunk_paths.append(chunk_path)
        written_cells += int(chunk_adata.n_obs)

        print(
            f"Chunk {len(chunk_paths):03d}: rows {row_start:,}:{row_end:,}; "
            f"written cells {written_cells:,}/{EXPECTED_OUTPUT_SHAPE[0]:,}"
        )

        del block, filtered_block, obs_chunk, chunk_adata
        gc.collect()

    if written_cells != EXPECTED_OUTPUT_SHAPE[0]:
        raise AssertionError(
            f"Chunk files contain {written_cells} cells; expected "
            f"{EXPECTED_OUTPUT_SHAPE[0]}."
        )
    if not chunk_paths:
        raise RuntimeError("No output chunks were created.")

    concat_on_disk(
        [str(path) for path in chunk_paths],
        str(temporary_output),
        axis=0,
        join="inner",
        merge="same",
        uns_merge="first",
        index_unique=None,
        max_loaded_elems=CONCAT_MAX_LOADED_ELEMS,
    )

    if not temporary_output.exists():
        raise FileNotFoundError(
            f"On-disk concatenation did not create {temporary_output}"
        )

    # Duplicate the sparse X group directly inside HDF5. This avoids a
    # multi-gigabyte in-memory X.copy() while producing a normal counts layer.
    with h5py.File(temporary_output, "r+") as handle:
        layers = handle.require_group("layers")
        if "counts" in layers:
            del layers["counts"]
        handle.copy("X", layers, name="counts")
        handle.flush()

finally:
    # Chunk files are no longer needed after concat_on_disk. If concatenation
    # fails, they are retained only until this cleanup block runs.
    if staging_dir.exists():
        shutil.rmtree(staging_dir)

# Release the source file before plotting and final validation.
adata.file.close()
del adata
gc.collect()

print("Temporary final object assembled:", temporary_output)


Assembling the final DS3 object on disk
Free disk space: 23.54 GiB
Required free-space guard: 20.00 GiB
Chunk 001: rows 0:4,096; written cells 3,853/218,735
Chunk 002: rows 4,096:8,192; written cells 7,682/218,735
Chunk 003: rows 8,192:12,288; written cells 11,504/218,735
Chunk 004: rows 12,288:16,384; written cells 15,357/218,735
Chunk 005: rows 16,384:20,480; written cells 19,230/218,735
Chunk 006: rows 20,480:24,576; written cells 23,075/218,735
Chunk 007: rows 24,576:28,672; written cells 26,892/218,735
Chunk 008: rows 28,672:32,768; written cells 30,679/218,735
Chunk 009: rows 32,768:36,864; written cells 34,437/218,735
Chunk 010: rows 36,864:40,960; written cells 38,204/218,735
Chunk 011: rows 40,960:45,056; written cells 41,956/218,735
Chunk 012: rows 45,056:49,152; written cells 45,817/218,735
Chunk 013: rows 49,152:53,248; written cells 49,648/218,735
Chunk 014: rows 53,248:57,344; written cells 53,187/218,735
Chunk 015: rows 57,344:61,440; written cells 56,175/218,735
Chunk 0

In [9]:
condition_composition = (
    final_obs.groupby("group", observed=True)
    .size()
    .rename("cells")
    .reset_index()
    .sort_values("group")
)
condition_composition.to_csv(
    AUDIT_DIR / "GSE182109_postQC_condition_composition.tsv",
    sep="\t",
    index=False,
)
condition_composition


,group,cells
0,LGG,13487
1,ndGBM,112416
2,rGBM,92832


In [10]:
# Plot only a deterministic subset of cells. This changes visualization
# density only; filtering and all reported counts use the complete dataset.
def deterministic_plot_obs(frame: pd.DataFrame, max_cells: int) -> pd.DataFrame:
    if len(frame) <= max_cells:
        return frame.copy()
    return frame.sample(n=max_cells, random_state=RANDOM_SEED).copy()


plot_columns = ["n_genes_by_counts", "total_counts", "pct_counts_mt"]
before_plot_obs = deterministic_plot_obs(input_obs[plot_columns], PLOT_MAX_CELLS)
after_plot_obs = deterministic_plot_obs(final_obs[plot_columns], PLOT_MAX_CELLS)

before_plot_adata = ad.AnnData(obs=before_plot_obs)
after_plot_adata = ad.AnnData(obs=after_plot_obs)

sc.pl.violin(
    before_plot_adata,
    plot_columns,
    jitter=0.4,
    multi_panel=True,
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE182109_QC_metrics_before_filtering.png")

sc.pl.violin(
    after_plot_adata,
    plot_columns,
    jitter=0.4,
    multi_panel=True,
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE182109_QC_metrics_after_filtering.png")

sc.pl.scatter(
    before_plot_adata,
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mt",
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE182109_scatter_before_filtering.png")

sc.pl.scatter(
    after_plot_adata,
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mt",
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE182109_scatter_after_filtering.png")

print(
    "QC figures saved using deterministic samples of up to",
    f"{PLOT_MAX_CELLS:,} cells per panel.",
)


QC figures saved using deterministic samples of up to 50,000 cells per panel.


In [11]:
with open(
    AUDIT_DIR / "GSE182109_QC_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(qc_summary, handle, indent=2)

print(qc_summary)


{'dataset_id': 'DS3_GSE182109', 'geo_accession': 'GSE182109', 'input_cells': 264951, 'input_genes': 38224, 'post_primary_filter_cells': 219418, 'post_QC_cells': 218735, 'post_QC_genes': 30284, 'retention_fraction': 0.8255677464889735, 'min_genes_exclusive': 500, 'max_genes_exclusive': 8000, 'min_counts_exclusive': 1000, 'primary_max_counts_exclusive': 60000, 'max_mt_percent_exclusive': 12.0, 'final_max_counts_inclusive': 45000, 'final_min_cells_per_gene': 10, 'raw_counts_in_X': True, 'raw_counts_layer': 'counts', 'normalization_applied': False, 'memory_safe_chunked_processing': True, 'timestamp': '2026-08-04T00:47:48', 'output_h5ad': 'C:\\Users\\AmirSilco\\Documents\\GitHub\\glioma-cnv-single-cell-atlas\\data\\processed\\DS3_GSE182109\\GSE182109_postQC.h5ad'}


In [13]:
from copy import deepcopy
from pathlib import Path
import gc

import anndata as ad
import h5py
import scanpy as sc


print("Running DS3 final metadata-repair and validation cell")


# ------------------------------------------------------------------
# Resolve final and temporary output paths
# ------------------------------------------------------------------
OUTPUT_H5AD = Path(OUTPUT_H5AD)

temporary_path = Path(
    globals().get(
        "temporary_output",
        OUTPUT_H5AD.with_name(
            f"{OUTPUT_H5AD.stem}.temporary{OUTPUT_H5AD.suffix}"
        ),
    )
)

metadata_sidecar = OUTPUT_H5AD.with_name(
    f"{OUTPUT_H5AD.stem}.metadata_repair{OUTPUT_H5AD.suffix}"
)


# Close any backed object left by the previous failed validation.
if "check" in globals():
    try:
        check.file.close()
    except Exception:
        pass

    del check
    gc.collect()


# If the assembled temporary file still exists, promote it.
# If the preceding failed cell already promoted it, use OUTPUT_H5AD directly.
if temporary_path.exists():
    temporary_path.replace(OUTPUT_H5AD)

if not OUTPUT_H5AD.exists():
    raise FileNotFoundError(
        "Neither the final output nor the assembled temporary output exists. "
        f"Expected final path: {OUTPUT_H5AD}"
    )


# ------------------------------------------------------------------
# Reconstruct the complete uns mapping
# ------------------------------------------------------------------
if "output_uns" in globals():
    uns_to_write = deepcopy(output_uns)

elif "input_uns" in globals() and "qc_summary" in globals():
    uns_to_write = deepcopy(input_uns)
    uns_to_write["dataset_level_QC"] = deepcopy(qc_summary)

elif "qc_summary" in globals():
    uns_to_write = {
        "dataset_level_QC": deepcopy(qc_summary)
    }

else:
    raise RuntimeError(
        "Could not reconstruct uns metadata because output_uns, "
        "input_uns, and qc_summary are unavailable."
    )


if "dataset_level_QC" not in uns_to_write:
    if "qc_summary" not in globals():
        raise RuntimeError(
            "uns metadata lacks 'dataset_level_QC' and qc_summary "
            "is unavailable."
        )

    uns_to_write["dataset_level_QC"] = deepcopy(qc_summary)


# ------------------------------------------------------------------
# Encode uns with AnnData, then copy it into the existing large H5AD
# without loading the expression matrix into RAM
# ------------------------------------------------------------------
if metadata_sidecar.exists():
    metadata_sidecar.unlink()

metadata_holder = ad.AnnData(
    uns=uns_to_write
)

metadata_holder.write_h5ad(
    metadata_sidecar,
    compression="lzf",
)

del metadata_holder
gc.collect()


try:
    with h5py.File(metadata_sidecar, "r") as source_handle:
        if "uns" not in source_handle:
            raise AssertionError(
                "Metadata sidecar does not contain an encoded /uns group."
            )

        with h5py.File(OUTPUT_H5AD, "r+") as output_handle:
            if "uns" in output_handle:
                del output_handle["uns"]

            source_handle.copy(
                "uns",
                output_handle,
                name="uns",
            )

            output_handle.flush()

finally:
    if metadata_sidecar.exists():
        metadata_sidecar.unlink()


# ------------------------------------------------------------------
# Validate the repaired final output
# ------------------------------------------------------------------
check = sc.read_h5ad(
    OUTPUT_H5AD,
    backed="r",
)

try:
    observed_shape = tuple(check.shape)
    expected_shape = tuple(EXPECTED_OUTPUT_SHAPE)

    if observed_shape != expected_shape:
        raise AssertionError(
            "Saved-file validation failed: "
            f"expected {expected_shape}, observed {observed_shape}"
        )

    if "counts" not in check.layers:
        raise AssertionError(
            "Saved output is missing layers['counts']."
        )

    if tuple(check.layers["counts"].shape) != expected_shape:
        raise AssertionError(
            "Saved counts layer has an unexpected shape: "
            f"{check.layers['counts'].shape}"
        )

    if check.obs_names.has_duplicates:
        raise AssertionError(
            "Saved output contains duplicated cell identifiers."
        )

    if check.var_names.has_duplicates:
        raise AssertionError(
            "Saved output contains duplicated gene identifiers."
        )

    if "dataset_level_QC" not in check.uns:
        raise AssertionError(
            "Saved output is still missing "
            "uns['dataset_level_QC']."
        )

    saved_qc = check.uns["dataset_level_QC"]

    required_qc_keys = [
        "dataset_id",
        "input_cells",
        "post_QC_cells",
        "post_QC_genes",
        "raw_counts_layer",
    ]

    missing_qc_keys = [
        key
        for key in required_qc_keys
        if key not in saved_qc
    ]

    if missing_qc_keys:
        raise AssertionError(
            "Saved uns['dataset_level_QC'] is missing keys: "
            f"{missing_qc_keys}"
        )

    if int(saved_qc["post_QC_cells"]) != expected_shape[0]:
        raise AssertionError(
            "Saved QC metadata reports an unexpected "
            "post-QC cell count."
        )

    if int(saved_qc["post_QC_genes"]) != expected_shape[1]:
        raise AssertionError(
            "Saved QC metadata reports an unexpected "
            "post-QC gene count."
        )

    required_obs_columns = [
        "n_genes_by_counts",
        "total_counts",
        "pct_counts_mt",
        "n_counts",
    ]

    missing_obs_columns = [
        column
        for column in required_obs_columns
        if column not in check.obs.columns
    ]

    if missing_obs_columns:
        raise AssertionError(
            "Saved output is missing obs columns: "
            f"{missing_obs_columns}"
        )

finally:
    check.file.close()


print()
print("Saved and validated:", OUTPUT_H5AD)
print("Validated shape:", expected_shape)
print("Validated layer: counts")
print("Validated uns entry: dataset_level_QC")

Running DS3 final metadata-repair and validation cell

Saved and validated: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\processed\DS3_GSE182109\GSE182109_postQC.h5ad
Validated shape: (218735, 30284)
Validated layer: counts
Validated uns entry: dataset_level_QC
